In [1]:
import polars as pl
import json

# Create table of places

## Load data for places

In [2]:
places = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places.csv"
)
places_to_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_types.csv"
)
feature_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/feature_types.csv"
)
event_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_types.csv"
)
locs = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/locations.csv"
)
places_to_attestations = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_attestation.csv"
)
places_to_establishment = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_establishment.csv",
    ignore_errors=True,
)
feature_class = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/feature_class.csv"
)
# Remove duplicate place entries
locs = locs.unique(subset=["place_id"], keep="first")

## Join tables to create places table

In [3]:
from re import sub


pt_df = places.join(places_to_types, on="place_id", how="left")
pt_df = pt_df.join(locs, on="place_id", how="left").unique(
    subset=["place_id"], keep="first"
)
pt_df = pt_df.rename({"ft_id": "yft_id"})
pt_df = pt_df.join(feature_types, on="yft_id", how="left", suffix="_ft_tp")
pt_df = pt_df.join(places_to_attestations, on="place_id", how="left")
pt_df = pt_df.join(places_to_establishment, on="place_id", how="left")
pt_df = pt_df.join(feature_class, on="fct_id", how="left")
pt_df = pt_df.select(
    [
        "place_id",
        "tr_title",
        "ch_pinyin",
        "latitude",
        "longitude",
        "attestation",
        "ch_title",
        "feature_class",
        "en_title",
        "est_year",
    ]
)
pt_df = pt_df.unique()
pt_df.head()

place_id,tr_title,ch_pinyin,latitude,longitude,attestation,ch_title,feature_class,en_title,est_year
str,str,str,f64,f64,str,str,str,str,i64
"""yrdb4546""","""甘泉堡""","""ganquan bao""",35.733998,105.171368,"""Fortifications_Year741_44""",null,null,null,741
"""yrdb2193""","""百花川""","""baihua chuan""",34.32195,106.340515,"""us_80545""","""川""","""natural features""","""creek""",1820
"""yrdb1124""","""張洪鎮""","""zhanghong zhen""",35.044556,108.277029,"""us_80198""","""鎮""","""habitations""","""town""",1820
"""yrdb3435""","""魚河堡""","""yuhe bao""",37.89519,109.851575,"""us_70331""","""堡""","""habitations""","""fortress""",1582
"""yrdb3254""","""青田""","""qingtian""",null,null,"""ds_1131""",null,null,null,null


In [4]:
print(places.shape)
print(pt_df.shape)
pt_df = pt_df.sort("est_year", nulls_last=True).unique(
    subset=["place_id"], keep="first"
)
print(pt_df.shape)

(4480, 3)
(5074, 10)
(4480, 10)


## Split into Upstream and Downstream data

In [5]:
# upstream = pt_df.filter(pl.col("attestation").str.starts_with("us"))#
# downstream = pt_df.filter(pl.col("attestation").str.starts_with("ds"))

In [6]:
upstream = pt_df.filter(
    pl.col("attestation")
      .is_not_null()
      .and_(
          pl.col("attestation")
            .str.to_lowercase()
            .str.contains("us_|fort")
      )
)

downstream = pt_df.filter(
    pl.col("attestation")
      .is_not_null()
      .and_(
          pl.col("attestation")
            .str.to_lowercase()
            .str.contains("ds_")
      )
)

In [7]:
places_to_attestations["attestation_id"].count()

5731

In [48]:
total_places_w_att = places_to_attestations["attestation_id"].count()

In [49]:
num_ds = places_to_attestations["attestation_id"].str.contains("us").sum()

In [50]:
num_us = places_to_attestations["attestation_id"].str.contains("ds").sum()

In [51]:
num_ds + num_us == total_places_w_att

False

In [52]:
total_places_w_att - (num_ds + num_us)

875

In [53]:
downstream_map = {y: 0 for y in downstream["est_year"].unique()}
for y in downstream["est_year"]:
    if y in downstream_map:
        downstream_map[y] += 1
    else:
        downstream_map[y] = 1
downstream_map

{None: 1527,
 -256: 4,
 327: 1,
 497: 1,
 612: 1,
 741: 1,
 1111: 16,
 1582: 4,
 1820: 1}

In [54]:
total = {y: 0 for y in upstream["est_year"].unique()}
for y in pt_df["est_year"]:
    if y in total:
        total[y] += 1
    else:
        total[y] = 1
total

{None: 1530,
 -256: 142,
 -206: 19,
 9: 73,
 220: 89,
 262: 32,
 263: 1,
 264: 1,
 281: 50,
 327: 10,
 366: 4,
 382: 3,
 395: 4,
 497: 121,
 546: 6,
 572: 14,
 612: 54,
 741: 157,
 830: 1,
 1111: 585,
 1189: 65,
 1330: 85,
 1449: 1,
 1582: 325,
 1820: 847}

In [55]:
upstream_map = {y: 0 for y in upstream["est_year"].unique()}
for y in upstream["est_year"]:
    if y in upstream_map:
        upstream_map[y] += 1
    else:
        upstream_map[y] = 1
upstream_map

{None: 3,
 -256: 138,
 -206: 19,
 9: 73,
 220: 89,
 262: 32,
 263: 1,
 264: 1,
 281: 50,
 327: 9,
 366: 4,
 382: 3,
 395: 4,
 497: 120,
 546: 6,
 572: 14,
 612: 53,
 741: 156,
 830: 1,
 1111: 569,
 1189: 65,
 1330: 85,
 1449: 1,
 1582: 321,
 1820: 846}

In [56]:
import requests

# Join Upstream places with infromation
eras = requests.get(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/eras.geojson"
).json()["features"]
dyns = requests.get(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/dynasties.geojson"
).json()["features"]
dyns_df = pl.DataFrame([f["properties"] for f in dyns], orient="row")
eras_df = pl.DataFrame([f["properties"] for f in eras], orient="row")
dyns_df.head()

id,name_en,name_ch,start_cert,start_date,end_cert,end_date
i64,str,str,str,i64,str,i64
1,"""Xia dynasty ""","""夏朝""","""n""",-2070,"""n""",-1600
2,"""Shang dynasty ""","""商朝 ""","""n""",-1600,"""n""",-1046
3,"""Zhou dynasty ""","""周朝 ""","""n""",-1046,"""y""",-256
4,"""Qin dynasty ""","""秦朝 ""","""y""",-221,"""y""",-207
5,"""Han dynasty ""","""漢朝 ""","""y""",-202,"""y""",220


In [57]:
eras_df.head()

id,monarch,era_name_en,era_name_ch,era_start,era_end
str,str,str,str,i64,i64
"""I0019""","""I0001""","""Jianlong""","""建隆""",960,963
"""I0020""","""I0001""","""Qiande""","""乾德""",963,968
"""I0021""","""I0001""","""Kaibao""","""開寶""",968,976
"""I0022""","""I0002""","""Taipingxingguo""","""太平興國""",976,984
"""I0023""","""I0002""","""Yongxi""","""雍熙""",984,987


In [58]:
dyns_df = dyns_df.select(["name_en", "name_ch", "start_date", "end_date"]).rename(
    {"name_en": "dynasty_en", "name_ch": "dynasty_ch"}
)

In [59]:
dyn_en = []
dyn_ch = []
for row in upstream.iter_rows():
    est_year = row[9]
    if est_year is None:
        dyn_en.append(None)
        dyn_ch.append(None)
        continue
    matched_dyn = dyns_df.filter(
        (pl.col("start_date") <= est_year) & (pl.col("end_date") >= est_year)
    )
    if matched_dyn.is_empty():
        dyn_en.append(None)
        dyn_ch.append(None)
    else:
        dyn_en.append(matched_dyn[0, "dynasty_en"].strip())
        dyn_ch.append(matched_dyn[0, "dynasty_ch"].strip())

upstream = upstream.with_columns(
    [pl.Series("dynasty_en", dyn_en), pl.Series("dynasty_ch", dyn_ch)]
)

upstream.head()

place_id,tr_title,ch_pinyin,latitude,longitude,attestation,ch_title,feature_class,en_title,est_year,dynasty_en,dynasty_ch
str,str,str,f64,f64,str,str,str,str,i64,str,str
"""yrdb1273""","""新興鋪""","""xinxing pu""",34.687664,107.076579,"""us_80274""","""鋪""","""habitations""","""post station""",741,"""Tang dynasty""","""唐朝"""
"""yrdb351""","""博望苑""","""bowang yuan""",34.211533,108.895655,"""us_20087""","""苑""","""habitations""","""park""",9,"""Han dynasty""","""漢朝"""
"""yrdb4399""","""牛皮關""","""niupi guan""",40.158763,113.440376,"""Fortifications_Year1111_95""",null,null,null,1111,"""Liao dynasty""","""遼朝"""
"""yrdb3811""","""子亭鎮""","""ziting zhen""",39.392868,94.939339,"""Fortifications_Year1111_147""",null,null,null,1111,"""Liao dynasty""","""遼朝"""
"""yrdb2826""","""趙堡鎮""","""zhaobao zhen""",34.878503,113.154137,"""us_80036""","""鎮""","""habitations""","""town""",1820,"""Qing dynasty""","""清朝"""


In [60]:
# convert upstream to json and write to upstream-data.json

upstream_data = []

for row in upstream.iter_rows():
    id = row[0]
    hz = row[1]
    py = row[2]
    x_coor = row[4]
    y_coor = row[3]
    date = row[9]
    regime = row[10]
    regime_ch = row[11]
    name_type = row[6]
    name_type_en = row[8]
    name_class_en = row[7]

    place_json = {
        "id": id,
        "hz": hz,
        "py": py,
        "x_coor": x_coor,
        "y_coor": y_coor,
        "date": date,
        "regime": regime,
        "regime_ch": regime_ch,
        "name_type": name_type,
        "name_type_en": name_type_en,
        "name_class_en": name_class_en,
    }
    upstream_data.append(place_json)

with open("upstream-data.json", "w", encoding="utf-8") as f:
     json.dump(upstream_data, f, ensure_ascii=False, indent=4)

# Create table for events

## Load data for events

In [61]:
import numpy as np

events = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events.csv"
)
event_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/event_types.csv"
)
event_cats = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/event_categories.csv"
)

events_to_places = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_places.csv"
).rename({"event": "event_id"})
events_to_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_types.csv"
)
events_to_sources = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/sources_to_events.csv",
    ignore_errors=True,
)
sources = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/sources.csv",
    null_values=["", "nan", "NaN", "NA", "null"],
    ignore_errors=True,
)
events_to_sources.head()

src_to_evt_id,source_id,event_id
str,i64,str
"""srev_1""",11001001,"""ev_1"""
"""srev_2""",11001002,"""ev_2"""
"""srev_3""",11001003,"""ev_3"""
"""srev_4""",11001004,"""ev_4"""
"""srev_5""",11001005,"""ev_5"""


In [62]:
events.head(1)

event_id,ch_date,western_date,description,notes
str,str,f64,str,str
"""ev_1""","""史前时代""",-2356.0,null,null


In [63]:
event_types.head(1)

event_type_id,zh_ch_title,en_title,en_type,evc_id,description
str,str,str,str,str,str
"""evtype_1""","""溢""","""yi""","""Flood""","""evc_1""","""Any time there is a zhang 漲(ra…"


In [64]:
events_to_places.head(1)

evtp_id,place_id,event_id,attestation
str,str,str,str
"""evtp_1""","""yrdb2""","""ev_858""","""ds_1019"""


In [65]:
events_to_types.head(1)

evetotyp_id,event_id,event_type_id
str,str,str
"""evetotyp_1""","""ev_1""","""evtype_1"""


In [66]:
event_cats.head(1)

evc_id,zh_cn_category,en_category
str,str,str
"""evc_1""","""水災""","""Disasters"""


In [67]:
sources.head(1)

source_id,source,page,chinese_date,western_date,old_placename_chinese,modern_placename_chinese,event_type_chinese,event_name,event_description,primary_source_1,primary_source_2,notes
i64,str,str,str,str,str,str,str,str,str,str,str,str
10100001,"""HDSJ""",null,"""传说时代""","""约-21世纪初""",null,null,null,"""大禹治水""","""传说中的尧舜时代，黄河流域发生大洪水，为制止洪水泛滥，尧召集…",null,null,null


In [68]:
print(f"events length: {events.shape}")
ev_df = events.join(events_to_types, on="event_id", how="left", suffix="_ett")
ev_df = ev_df.join(event_types, on="event_type_id", how="left")
print(f"ev_df length: {ev_df.shape}")
# wanted columsn from event_types: zh_ch_title,	en_title, en_type
# wanted columns from events: event_id	ch_date	western_date	description	notes
ev_df = ev_df.select(
    [
        "event_id",
        "ch_date",
        "western_date",
        "description",
        "notes",
        "zh_ch_title",
        "en_title",
        "evc_id",
        "en_type",
    ]
)
ev_df = ev_df.rename(
    {
        "ch_date": "event_date_ch",
        "western_date": "event_date_western",
        "description": "event_description",
        "notes": "event_notes",
        "zh_ch_title": "event_type_ch",
        "en_title": "event_type_py",
        "en_type": "event_type_en",
    }
)
print(f"ev_df length: {ev_df.shape}")
ev_df.head()

events length: (3754, 5)
ev_df length: (5349, 12)
ev_df length: (5349, 9)


event_id,event_date_ch,event_date_western,event_description,event_notes,event_type_ch,event_type_py,evc_id,event_type_en
str,str,f64,str,str,str,str,str,str
"""ev_1""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_2""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""災""","""zai""","""evc_1""","""Disaster"""
"""ev_4""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""


In [69]:
ev_df = ev_df.join(event_cats, on="evc_id", how="left")
ev_df.head()

event_id,event_date_ch,event_date_western,event_description,event_notes,event_type_ch,event_type_py,evc_id,event_type_en,zh_cn_category,en_category
str,str,f64,str,str,str,str,str,str,str,str
"""ev_1""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_2""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""災""","""zai""","""evc_1""","""Disaster""","""水災""","""Disasters"""
"""ev_4""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""


In [70]:
ev_df = ev_df.rename(
    {"zh_cn_category": "type_category_ch", "en_category": "type_category_en"}
)

In [71]:
# group by event_id and aggregate types columns into lists
ev_df = ev_df.group_by(["event_id", "event_date_western"]).agg(
    pl.col("event_date_ch").first(),
    pl.col("event_description").first(),
    pl.col("event_notes").first(),
    pl.col("event_type_ch").implode(),
    pl.col("event_type_py").implode(),
    pl.col("event_type_en").implode(),
    pl.col("evc_id").implode(),
    pl.col("type_category_ch").implode(),
    pl.col("type_category_en").implode(),
)
print(f"original events length: {events.shape}")
print(f"ev_df length after groupby: {ev_df.shape}")
ev_df.head()

original events length: (3754, 5)
ev_df length after groupby: (3754, 11)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,type_category_ch,type_category_en
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str]
"""ev_2375""",1675.0,"""康熙十四年""",null,"""《黄河年表》引《清河县志》作清河决口""","[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_3337""",1828.0,"""清宣宗道光八年""",null,null,"[""修""]","[""xiu""]","[""Repair of Structures""]","[""evc_2""]","[""水利""]","[""Management""]"
"""ev_1508""",1348.0,"""元至正八年""",null,null,"[""決"", ""徙""]","[""jue"", ""xi""]","[""Breach"", ""Course change""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_180""",54.0,"""光武建武三十年""",null,null,"[""溢""]","[""yi""]","[""Flood""]","[""evc_1""]","[""水災""]","[""Disasters""]"
"""ev_1717""",1461.0,"""明天顺五年""",null,null,"[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"


In [72]:
# Join Sources
print(f"original events length: {events.shape}")
ev_df = ev_df.join(events_to_sources, on="event_id", how="left")
ev_df = ev_df.join(sources, on="source_id", how="left")
# ev_df columns: event_id, event_date_western, event_date_ch	event_description, event_notes, event_type_ch, event_type_py, event_type_en
# source columns: source, page, chinese_date, western_date, old_placename_chinese, modern_placename_chinese, event_type_chinese, event_name, event_description, primary_source_1, primary_source_2, notes
ev_df = ev_df.rename(
    {
        "page": "source_page",
        "chinese_date": "source_ch_date",
        "western_date": "source_western_date",
        "event_type_chinese": "source_event_type_chinese",
        "event_name": "source_event_name",
        "event_description_right": "source_event_description",
    }
)
ev_df.head()

original events length: (3754, 5)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,type_category_ch,type_category_en,src_to_evt_id,source_id,source,source_page,source_ch_date,source_western_date,old_placename_chinese,modern_placename_chinese,source_event_type_chinese,source_event_name,source_event_description,primary_source_1,primary_source_2,notes
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""ev_2375""",1675.0,"""康熙十四年""",null,"""《黄河年表》引《清河县志》作清河决口""","[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]","""srev_3662""",10616012,"""SLSY""","""329""","""康熙十四年""","""1675""","""徐州、宿迁、睢宁等""",null,null,null,"""决徐州潘家塘，宿迁蔡家楼，又决睢宁花山坝，复灌清河治，民多流…",null,null,"""《黄河年表》引《清河县志》作清河决口"""
"""ev_2375""",1675.0,"""康熙十四年""",null,"""《黄河年表》引《清河县志》作清河决口""","[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]","""srev_3663""",10802194,"""ZDZH""","""297""","""清康熙十四年""","""1675""",null,"""江苏""","""黄河决溢""",null,"""决徐州潘家塘，宿迁蔡家楼，又决睢宁花山坝，复灌清河治，民多流…","""《清史稿·河渠志》""",null,null
"""ev_3337""",1828.0,"""清宣宗道光八年""",null,null,"[""修""]","[""xiu""]","[""Repair of Structures""]","[""evc_2""]","[""水利""]","[""Management""]","""srev_4727""",10412244,"""HHNB""",null,"""清宣宗道光八年""","""1828""",null,null,"""修""",null,"""三月奏挑培王家山天然闸下河堰。（淮系年表）""",null,null,null
"""ev_1508""",1348.0,"""元至正八年""",null,null,"[""決"", ""徙""]","[""jue"", ""xi""]","[""Breach"", ""Course change""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]","""srev_2371""",10805012,"""ZDZH""","""366""","""元至正八年""","""1348""",null,"""山东""","""黄河凌汛""",null,"""正月辛亥，河决，陷济宁路。""","""《元史·五行志》""",null,null
"""ev_1508""",1348.0,"""元至正八年""",null,null,"[""決"", ""徙""]","[""jue"", ""xi""]","[""Breach"", ""Course change""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]","""srev_2372""",10300360,"""LYZS""","""404""","""至正八年""","""1348""",null,null,null,"""黄河流域大水决溢""","""正月河决，陷济宁路，迁路于济州. 六月山东大水民饥""","""《元史·本纪·五行志》""",null,"""欧阳玄：《至正河防记》"""


In [73]:
ev_df = ev_df.select(
    [
        "event_id",
        "event_date_western",
        "event_date_ch",
        "event_description",
        "event_notes",
        "event_type_ch",
        "event_type_py",
        "event_type_en",
        "evc_id",
        "source",
        "source_page",
        "source_ch_date",
        "source_western_date",
        "source_event_type_chinese",
        "source_event_name",
        "source_event_description",
        "type_category_ch",
        "type_category_en",
    ]
)
print(f"ev_df length after join2: {ev_df.shape}")
ev_df.head()

ev_df length after join2: (5235, 18)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,type_category_ch,type_category_en
str,f64,str,str,str,list[str],list[str],list[str],list[str],str,str,str,str,str,str,str,list[str],list[str]
"""ev_2375""",1675.0,"""康熙十四年""",null,"""《黄河年表》引《清河县志》作清河决口""","[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","""SLSY""","""329""","""康熙十四年""","""1675""",null,null,"""决徐州潘家塘，宿迁蔡家楼，又决睢宁花山坝，复灌清河治，民多流…","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_2375""",1675.0,"""康熙十四年""",null,"""《黄河年表》引《清河县志》作清河决口""","[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","""ZDZH""","""297""","""清康熙十四年""","""1675""","""黄河决溢""",null,"""决徐州潘家塘，宿迁蔡家楼，又决睢宁花山坝，复灌清河治，民多流…","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_3337""",1828.0,"""清宣宗道光八年""",null,null,"[""修""]","[""xiu""]","[""Repair of Structures""]","[""evc_2""]","""HHNB""",null,"""清宣宗道光八年""","""1828""","""修""",null,"""三月奏挑培王家山天然闸下河堰。（淮系年表）""","[""水利""]","[""Management""]"
"""ev_1508""",1348.0,"""元至正八年""",null,null,"[""決"", ""徙""]","[""jue"", ""xi""]","[""Breach"", ""Course change""]","[""evc_1"", ""evc_1""]","""ZDZH""","""366""","""元至正八年""","""1348""","""黄河凌汛""",null,"""正月辛亥，河决，陷济宁路。""","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_1508""",1348.0,"""元至正八年""",null,null,"[""決"", ""徙""]","[""jue"", ""xi""]","[""Breach"", ""Course change""]","[""evc_1"", ""evc_1""]","""LYZS""","""404""","""至正八年""","""1348""",null,"""黄河流域大水决溢""","""正月河决，陷济宁路，迁路于济州. 六月山东大水民饥""","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"


In [74]:
# Group and aggregate sources into lists
ev_df = ev_df.group_by(
    [
        "event_id",
        "event_date_western",
        "event_date_ch",
        "event_description",
        "event_notes",
        "evc_id",
        "event_type_ch",
        "event_type_py",
        "event_type_en",
        "type_category_ch",
        "type_category_en",
    ]
).agg(
    [
        pl.col("source").implode(),
        pl.col("source_page").implode(),
        pl.col("source_ch_date").implode(),
        pl.col("source_western_date").implode(),
        pl.col("source_event_type_chinese").implode(),
        pl.col("source_event_name").implode(),
        pl.col("source_event_description").implode(),
    ]
)

In [75]:
# Create formatted citation from paired lists
ev_df1 = ev_df.with_columns(
    [
        pl.struct(
            [
                "source",
                "source_western_date",
                "source_ch_date",
                "source_event_description",
                "source_page",
            ]
        )
        .map_elements(
            lambda x: "; ".join(
                [
                    f"""{name}{', in ' + west if west is not None else ''} {'('+ch+')' if ch is not None else ''} {'describes it as ' + description if description is not None else ''} {'(p. ' + page + ')' if page is not None else ''}""".strip()
                    for name, west, ch, description, page in zip(
                        x["source"],
                        x["source_western_date"],
                        x["source_ch_date"],
                        x["source_event_description"],
                        x["source_page"],
                    )
                    if name is not None
                ]
            ),
            return_dtype=pl.String,
        )
        .alias("citation")
    ]
)
print(f"ev_df length after citations: {ev_df1.shape}")
ev_df1.head()

ev_df length after citations: (3754, 19)


event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str
"""ev_2718""",1734.0,"""清世宗雍正十二年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""HHNB""]",[null],"[""清世宗雍正十二年""]","[""1734""]","[""溢""]",[null],"[""黄水溢，宿州水。（淮系年表）""]","""HHNB, in 1734 (清世宗雍正十二年) descr…"
"""ev_3149""",1808.0,"""清仁宗嘉庆十三年""",null,null,"[""evc_2""]","[""放""]","[""fang""]","[""Dam/Sluice Opening""]","[""水利""]","[""Management""]","[""HHNB""]",[null],"[""清仁宗嘉庆十三年""]","[""1808""]","[""修""]",[null],"[""二月初二日陈家浦开放引河，初十日合龙。（南河成案续编）""]","""HHNB, in 1808 (清仁宗嘉庆十三年) descr…"
"""ev_103""",-185.0,"""高后三年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""ZHTS""]","[""27""]","[""高后三年""]","[""-185""]",[null],[null],"[""夏，江水、汉水溢，流民四千余家（《汉书·高后纪》）夏，汉中、南郡大水，水出，流四千余家（《汉书·五行志》""]","""ZHTS, in -185 (高后三年) describes…"
"""ev_1092""",1118.0,"""宋徽宗重合元年""",null,null,"[""evc_2""]","[""治""]","[""zhi""]","[""Management""]","[""水利""]","[""Management""]","[""HHNB""]",[null],"[""宋徽宗重合元年""]","[""1118""]","[""治""]",[null],"[""三月己亥，诏：“滑州、浚州界万年堤，全藉林木固护堤岸，其广行种植，以壮地势。”（宋史河渠志）""]","""HHNB, in 1118 (宋徽宗重合元年) descri…"
"""ev_268""",283.0,"""晋武帝太康四年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""HHNB""]",[null],"[""晋武帝太康四年""]","[""283""]","[""大水""]",[null],"[""十二月河南大水。（晋书）""]","""HHNB, in 283 (晋武帝太康四年) describ…"


In [76]:
print(f"ev_df length: {ev_df.shape}")
ev_df2 = ev_df1.join(events_to_places, on="event_id", how="left", suffix="_etp")

print(f"ev_df2 len after join places: {ev_df2.shape}")

ev_df length: (3754, 18)
ev_df2 len after join places: (10350, 22)


In [77]:
ev_df2.filter(pl.col("place_id").is_not_null()).count()

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
9955,9955,9955,995,379,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955


In [78]:
ev_df2.filter(pl.col("attestation").str.starts_with("us")).count()

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [79]:
print(downstream.shape)
downstream = downstream.filter(
    (pl.col("latitude").is_not_null()).and_(pl.col("longitude").is_not_null())
)

(1556, 10)


In [80]:
ev_df2.head(1)

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,str,str
"""ev_2718""",1734.0,"""清世宗雍正十二年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""HHNB""]",[null],"[""清世宗雍正十二年""]","[""1734""]","[""溢""]",[null],"[""黄水溢，宿州水。（淮系年表）""]","""HHNB, in 1734 (清世宗雍正十二年) descr…","""evtp_2620""","""yrdb864""","""ds_1244"""


In [81]:
yrdb_events = []

for entry in downstream.iter_rows():
    yrdb_id = entry[0]
    tr_title = entry[1]
    ch_pinyin = entry[2]
    lat = entry[3]
    long = entry[4]
    class_en = entry[7]
    type_ch = entry[6]
    type_en = entry[8]
    events = []
    for event in ev_df2.filter(pl.col("place_id") == yrdb_id).iter_rows():
        event_id = event[0]
        en_date_start = event[1]
        ch_date = event[2]
        en_cat = event[10]
        en_type = event[8]
        en_title = event[7]
        ch_cat = event[9]
        ch_title = event[6]
        description = event[17]
        citation = event[18]
        source = event[11]
        src_page = event[12]
        src_ch_date = event[13]
        src_west_date = event[14]

        event_dict = {
            "event_id": event_id,
            "en_date_start": en_date_start,
            "ch_date": ch_date,
            "en_cat": en_cat,
            "en_type": en_type,
            "en_title": en_title,
            "ch_cat": ch_cat,
            "ch_title": ch_title,
            "description": description,
            "citation": citation,
            "source": source,
            "src_page": src_page,
            "src_ch_date": src_ch_date,
            "src_west_date": src_west_date,
        }
        events.append(event_dict)

    # Build place dictionary with nested events
    place_dict = {
        "yrdb_id": yrdb_id,
        "tr_title": tr_title,
        "ch_pinyin": ch_pinyin,
        "lat": lat,
        "long": long,
        "class_en": class_en,
        "type_ch": type_ch,
        "type_en": type_en,
        "events": events,
    }
    yrdb_events.append(place_dict)

# Write to JSON file
with open("yrdb_events.json", "w", encoding="utf-8") as f:
    json.dump(yrdb_events, f, ensure_ascii=False, indent=2)